# Week 8 · Reports, screens and a first chart

[![Open in Colab](../assets/badges/colab-badge.svg)](https://colab.research.google.com/github/loem-ms/programming-for-information-systems-site/blob/main/content/downloads/Week_08.ipynb)

The same records serve different people. This week you decide who sees what, put the most urgent food first, keep the numbers honest, and write your own web page.

| | |
|---|---|
| **This week's question** | Who needs which information, and how do we keep numbers honest and private data private? |
| **Python you will learn** | `sorted` with a named key function, choosing fields from a dictionary, adding amounts into a dictionary (`total_by`), two honest charts side by side, a web page from plain Python (`html.escape`) |
| **Principle 8** | **Right information for the right person.** Honest numbers, private data kept private. |
| **File you add** | `my_foodbridge/report.py` → `public_row`, `most_urgent_first`, `portions_by_status` · and your own page, `my_foodbridge/public_board.html` |
| **What you will see** | `bridge.show("board")`: the coordinator view and the public board side by side, most urgent first, with no contacts on the public board · **your own public page**, made by about 15 lines of your Python |

*Campus FoodBridge is fictional. All organisations, people and numbers are invented.*

![System map, week 8: report.py shows the right view to each person](../assets/diagrams/system_map_w08.svg)

### The story so far

Last week `storage.py` made your system remember: every change is saved safely, the history only grows, and a restart loads it all back.

This morning the coordinator copies the full records onto the canteen's public screen. Sunrise Bakery's contact email is there for everyone to read. The rejected bananas sit at the top, and the vegetable curry with only 4 hours left waits at the bottom. Then a sponsor adds the screen's three numbers together and asks why the team "rescued 166 meals".

The records are safe, but the wrong people see the wrong information, and the numbers mislead. A system also has to **show**: the right information, to the right person, in the right order, with honest numbers. That is this week's work.

## Before you start

1. **File → Save a copy in Drive.** Work in your copy.
2. Run the setup cell. In Colab it asks to connect Google Drive: say yes, so your `my_foodbridge/` folder is kept between weeks.
3. ⭐ = everyone · ⭐⭐ = stretch.

In [ ]:
# ▶ Run this cell first (setup)
# In Colab or Jupyter, this cell unpacks the FoodBridge kit next to the notebook.
# This web page is for reading. To run the notebook, use the 'Open in Colab' button at the top of this page.


## Class warm-up · Research 6 exchange (10 min)

In groups of three, share the AI answer you checked.

1. Which claim did you accept, which did you correct, and which would you still ask about?
2. Which original source did you use? How long did the check take?
3. What could have gone wrong if nobody had checked? Who would have been hurt?

📝 **My notes:** one checking habit I will use from now on …

## Session 1 · Learn and try (90 min)

### The idea: the right information for the right person

Think of a hospital. The doctor sees your full file: name, phone number and history. The waiting-room screen shows only your ticket number and the room. Same data, different **view**: each view is designed for one person and one task, and leaves out what that person should not see.

| View | Who reads it | What it needs | What it must not show |
|---|---|---|---|
| Coordinator view | the coordinator | every offer, with the donor and the **contact**, to arrange pickups | nothing is hidden |
| Public board | anyone in the canteen: students, donors, partners, strangers | what food is moving, and how much | contact details |

A view must also be **honest**. Run the cell and read the three numbers.

In [ ]:
records = bridge.sample_records()
offered = 0
picked_up = 0
delivered = 0
for record in records:
    offered = offered + record["portions"]
    if record["status"] in ["picked_up", "delivered"]:
        picked_up = picked_up + record["portions"]
    if record["status"] == "delivered":
        delivered = delivered + record["portions"]

print("Portions offered:  ", offered)
print("Portions picked up:", picked_up)
print("Portions delivered:", delivered)

100 portions were **offered**, 48 **picked up** and 18 **delivered**: three stages of the same food. F002's 18 portions are inside all three numbers, so adding them (166) counts them three times. Different measures are never added together. The unit is **portions**, not "meals": the system does not know how many people ate.

🤔 A sponsor asks: *"How much food did you rescue today?"* Which number would you give, and what would you say with it?

📝 **My answer:** …

### 1 · `sorted()` makes a new list in order

🤔 **Predict** the last line before you run the cell.

In [ ]:
marks = [72, 45, 90, 61]
print(sorted(marks))
print(sorted(marks, reverse=True))
print(sorted(marks, reverse=True)[:3])     # a slice from Week 7: the top three
print(marks)

**Prediction:** …  **Result:** …

`sorted` returns a **new** list; the original list does not change. `reverse=True` puts the largest first. This matters for screens: the coordinator view and the public board can sort the same records in different orders without disturbing each other.

### 2 · Sorting records with a named key function

Python cannot guess how to order dictionaries: by name? by mark? You tell it with a **key function**. It receives one item and returns the value to sort by.

🤔 **Predict** the order of the names.

In [ ]:
students = [{"name": "Sokha", "mark": 72},
            {"name": "Dara", "mark": 90},
            {"name": "Vanna", "mark": 61}]


def by_mark(student):
    return student["mark"]


for student in sorted(students, key=by_mark):
    print(student["name"], student["mark"])

**Prediction:** …  **Result:** …

Write `key=by_mark` **without** `( )`. You give `sorted` the function itself, and `sorted` calls it once for each student. A good name says the order: `by_mark`, `hours_left_of`.

### ✍️ Try it · `cheapest_names`

A shop lists its products, cheapest first. Write a key function `price_of(product)`, then `cheapest_names(products)`: return the product names, cheapest first.

In [ ]:
def price_of(product):
    # TODO: return the value to sort by
    return 0


def cheapest_names(products):
    names = []
    # TODO: for each product in sorted(products, key=price_of), append its "name" to names
    return names


shop = [{"name": "Soap", "riel": 4000},
        {"name": "Rice 5 kg", "riel": 18000},
        {"name": "Pen", "riel": 1000},
        {"name": "Notebook", "riel": 2500}]
bridge.check("cheapest_names", cheapest_names, examples=[
    (shop, ["Pen", "Notebook", "Soap", "Rice 5 kg"]),
    ([], []),
])

### 3 · Choosing fields: an allowed list

A public view is a **smaller copy** of a record. The safe way to build it: start with an empty dictionary and copy only the fields on an **allowed list**. (The phone number is invented.)

🤔 **Predict:** the last two lines add a new private field. Will the notice board show it?

In [ ]:
student = {"name": "Dara", "phone": "012 000 111", "mark": 90, "class": "MIS1"}
NOTICE_BOARD_FIELDS = ["name", "class"]


def notice_board_row(student):
    row = {}
    for field in NOTICE_BOARD_FIELDS:
        row[field] = student.get(field, "")
    return row


print(notice_board_row(student))

student["home_address"] = "Street 271"      # someone adds a new private field later
print(notice_board_row(student))

**Prediction:** …  **Result:** …

The other way, copying everything and then deleting the private fields, is risky. When someone adds a new private field later, it appears on the board until somebody notices. With an allowed list, a new field stays hidden until a person **decides** to show it. `student.get(field, "")` gives `""` when a field is missing, instead of stopping with a `KeyError`.

### ✍️ Try it · `seat_row`

The exam seat list on the classroom door shows only the seat and the name. Write `seat_row(student)`: return a new dictionary with only the fields in `SEAT_FIELDS`. Use `""` for a missing field.

In [ ]:
SEAT_FIELDS = ["seat", "name"]


def seat_row(student):
    row = {}
    # TODO: for each field in SEAT_FIELDS, copy student.get(field, "") into row
    return row


bridge.check("seat_row", seat_row, examples=[
    ({"name": "Dara", "phone": "012 000 111", "seat": "A12", "mark": 90}, {"seat": "A12", "name": "Dara"}),
    ({"name": "Sokha", "seat": "B03", "home_address": "Street 271"}, {"seat": "B03", "name": "Sokha"}),
    ({"name": "Vanna"}, {"seat": "", "name": "Vanna"}),
])

### 4 · Adding amounts into a dictionary

In Week 6 you counted into a dictionary: `counts[key] = counts.get(key, 0) + 1`. To add up **amounts** instead, add the amount, not 1. Values from a CSV file are text, so use `int()` first.

### ✍️ Try it · `total_by`

Write `total_by(rows, field, amount_field)`: for each value in `field`, add up the numbers in `amount_field`. It works for any table: a shop's sales, phone top-ups, or a semester of offers.

In [ ]:
def total_by(rows, field, amount_field):
    totals = {}
    # TODO: for each row, add int(row[amount_field]) to totals[row[field]]   (totals.get(..., 0) + ...)
    return totals


sales = [{"item": "bread", "qty": 2}, {"item": "rice", "qty": 1}, {"item": "bread", "qty": 3}]
csv_rows = [{"name": "Dara", "amount": "5000"}, {"name": "Dara", "amount": "1000"}]    # CSV values are text
bridge.check("total_by", total_by, examples=[
    ((sales, "item", "qty"), {"bread": 5, "rice": 1}),
    ((csv_rows, "name", "amount"), {"Dara": 6000}),
    (([], "item", "qty"), {}),
])

### 5 · Count or sum? Which donor matters most?

The kit holds last semester's 404 offers in `semester.csv` (invented). The cell reads it with the two lines from Week 5; `count_by` (Week 6) counts each donor's offers and **your** `total_by` adds up their portions.

In [ ]:
import csv
import matplotlib.pyplot as plt

with open("foodbridge_kit/data/semester.csv", encoding="utf-8") as file:
    offers = list(csv.DictReader(file))


def count_by(rows, field):                       # from Week 6
    counts = {}
    for row in rows:
        counts[row[field]] = counts.get(row[field], 0) + 1
    return counts


offers_by_donor = count_by(offers, "donor")
portions_by_donor = total_by(offers, "donor", "portions")      # your Try it
if portions_by_donor == {}:
    print("🔧 total_by is not finished yet: model numbers in use")
    portions_by_donor = {"Lotus Cafe": 766, "Sunrise Bakery": 1721, "Campus Canteen": 1780,
                         "Riverside Restaurant": 1292, "Night Market Vendor": 674,
                         "Green Market Stall": 1547, "Student Club Bake Sale": 379, "Hotel Kitchen": 827}
donors = list(offers_by_donor.keys())
offer_counts = []
portion_totals = []
for donor in donors:
    offer_counts.append(offers_by_donor[donor])
    portion_totals.append(portions_by_donor.get(donor, 0))
print(len(offers), "offers from", len(donors), "donors")

An honest chart has a **title** that says what is counted, an axis **label with the unit**, and bars that **start at 0**. `barh` draws the bars sideways, so long names fit. Replace the four `TODO` texts and run.

In [ ]:
figure, (left, right) = plt.subplots(1, 2, figsize=(11, 4))      # two charts side by side
left.barh(donors, offer_counts)
left.set_title("TODO: what is counted?")
left.set_xlabel("TODO: which unit?")
right.barh(donors, portion_totals)
right.set_title("TODO")
right.set_xlabel("TODO")
plt.tight_layout()
plt.show()

🤔 **Which donor matters most?** Campus Canteen sends the most offers (85) and portions (1,780). Hotel Kitchen sends the **fewest** offers (13) but 827 portions: about 64 per offer, three times the canteen's 21. To plan trips, count offers; to report food rescued, add portions. A dashboard must say **which** measure it shows.

📝 **My answer:** to plan trips I would use … · to thank donors for food rescued I would use … · because …

### 6 · The same numbers, a misleading picture

Your `total_by` also gives the portions for each status of the sample day. Chart B shows them with a y axis that starts at 15.

🤔 **Predict:** in chart B, how many times longer will "rejected" (40) look than "delivered" (18)? What happens to "approved" (12)?

In [ ]:
counts = total_by(bridge.sample_records(), "status", "portions")      # your Try it
if counts == {}:
    print("🔧 total_by is not finished yet: model numbers in use")
    counts = {"picked_up": 30, "delivered": 18, "rejected": 40, "approved": 12}
statuses = list(counts.keys())
portions = list(counts.values())

figure, (honest, misleading) = plt.subplots(1, 2, figsize=(10, 3))
honest.bar(statuses, portions)
honest.set_title("A · Portions by status (sample day)")
honest.set_ylabel("portions")
honest.set_ylim(bottom=0)            # starts at 0: bar lengths are honest
misleading.bar(statuses, portions)
misleading.set_title("B · Portions by status")
misleading.set_ylim(15, 42)          # starts at 15, not 0
plt.show()

**Prediction:** …  **Result:** …  **Why:** …

A bar's length is the message. 40 is about twice 18, but in chart B "rejected" looks eight times longer than "delivered", and the 12 approved portions vanish. No number changed, and the chart still misleads (B has no unit either).

These four bars can be compared because each offer has exactly **one** status; together they make the 100 portions offered. Offered, picked up and delivered are stages of the same food: never add them.

## Session 2 · Practise and judge (90 min)

*If you opened this notebook in a new session, run the setup cell at the top first so `bridge` is ready.*

### 🎫 This week's ticket

> **From:** the coordinator · **To:** you
> "The canteen wants a public screen. Last week I nearly put the full records on it, with every donor's email. I need a public board that shows only safe fields, the most urgent food at the top of both screens, and honest numbers for the chart."

| | |
|---|---|
| **File** | `my_foodbridge/report.py` |
| **Setting** | `PUBLIC_FIELDS`: the fields anyone may see (default `["id", "food", "category", "portions", "status"]`) |
| **Functions** | `public_row(record)` · `most_urgent_first(records)` · `portions_by_status(records)` |
| **How the system uses them** | the public board shows your `public_row` of each offer; both screens use your `most_urgent_first`; the chart and the totals use your `portions_by_status` |
| **Done when** | the three checks pass, and `bridge.show("board")` shows your parts ✅, F002 at the top, and no 🔓 warning (or your decision log explains why) |

| Call | Result |
|---|---|
| `public_row(record F001)` | `{"id": "F001", "food": "Bread rolls", "category": "bread", "portions": 30, "status": "picked_up"}`: a new dictionary, no donor, no contact |
| `most_urgent_first(4 sample records)` | a **new** list: F002 (2 hours left), F004 (4), F001 (5), F003 (20). The list you passed in is unchanged. |
| `portions_by_status(4 sample records)` | `{"picked_up": 30, "delivered": 18, "rejected": 40, "approved": 12}` |
| `portions_by_status([])` | `{}` |

### ✏️ 🔐 Model it on paper first: who may see which field? (15 min)

Decide on paper before any code. Copy the table and write ✅ (needs it for their task), ❌ (must not see it) or ❓ (only if the person agrees) in each box.

| Field | Coordinator | Volunteer | Partner | Public board |
|---|---|---|---|---|
| `id`, `food`, `portions`, `status` | ✅ | | | |
| `category` | | | | |
| `hours_left` | | | | |
| `donor` | | | | |
| `contact` | ✅ | | | ❌ |
| `partner` | | | | |

1. Compare with a partner. Where do you disagree? Who must agree for each ❓?
2. Your **Public board** column is the first draft of `PUBLIC_FIELDS`. Sketch that board with the four sample offers in the order a volunteer needs. Which is at the top, and why?

📝 **My matrix and sketch:** describe them here (or paste a photo into this text cell).

### 🧩 Ladder 1 · Put the lines in order

The team leader plans the vehicles: the open offers, biggest first. Click the lines into order. One line does not belong.

In [ ]:
bridge.parsons("vehicle_plan")

### 🧩 Ladder 2 · From a blank cell

The coordinator's history screen shows the newest event at the top, like the call log on a phone. Write `latest_first(events)`: return a new list of events, highest `"step"` first. Write your own key function above it (like `price_of` in Session 1).

In [ ]:
def latest_first(events):
    # Your code here.
    return []


day_events = [{"step": 1, "event": "added"}, {"step": 3, "event": "moved"}, {"step": 2, "event": "moved"}]
bridge.check("latest_first", latest_first, examples=[
    (day_events, [day_events[1], day_events[2], day_events[0]]),
    ([], []),
])

### ⚖️ Good or risky?

The coordinator asked an AI assistant: *"Make a dashboard for the public screen in the canteen."* It gave you **Version A**. **Version B** is a rewrite. Both run without errors.

**Version A** (from the AI assistant)

```python
def by_food_name(record):
    return record["food"]

def dashboard(records):
    offered = 0
    picked_up = 0
    delivered = 0
    for record in records:
        offered += record["portions"]
        if record["status"] == "picked_up":
            picked_up += record["portions"]
        if record["status"] == "delivered":
            delivered += record["portions"]
    lines = [f"Meals served: {offered + picked_up + delivered}"]
    for record in sorted(records, key=by_food_name):
        lines.append(f"{record['food']:<26} {record['portions']:>4}  {record['contact']}")
    return "\n".join(lines)
```

On the four sample records, Version A shows:

```text
Meals served: 148
Bananas                      40  greenstall@example.org
Bread rolls                  30  sunrise@example.org
Fried rice boxes             18  canteen@example.org
Vegetable curry with rice    12  lotus@example.org
```

**Version B**

```python
def by_hours_left(record):
    return record["hours_left"]

def public_board(records):
    delivered = 0
    for record in records:
        if record["status"] == "delivered":
            delivered += record["portions"]
    lines = [f"Portions delivered so far: {delivered}"]
    for record in sorted(records, key=by_hours_left):
        if record["status"] not in ["rejected", "cancelled"]:
            portions = record["portions"]
            lines.append(f"{record['food']:<26} {portions:>4} portions  {record['status']}")
    return "\n".join(lines)
```

```text
Portions delivered so far: 18
Fried rice boxes             18 portions  delivered
Vegetable curry with rice    12 portions  approved
Bread rolls                  30 portions  picked_up
```

(`"\n".join(lines)` joins a list of lines into one text, with a new line between them.)

Use **Principle 8 (right information for the right person)**. Find **at least three** problems in Version A. For each one: which line, and which offer or person is harmed? How would a test catch it?

📝 **Problems I found in Version A:**

1.
2.
3.

<details><summary>Instructor's view</summary>

1. **A dishonest number.** "Meals served: 148" adds three different measures: all portions offered (100), portions with status picked up (30) and portions delivered (18). F001's bread and F002's rice are each counted twice. F003's 40 portions of bananas were rejected and never left the market, but they are inside the 148. The unit is wrong too: the system counts **portions**, not meals, and it cannot know what was "served". The honest number is **18 portions delivered**. Sponsors who plan with 148 are misled, and the team loses trust as soon as someone checks.
2. **Private data on a public screen.** Every donor's contact email is shown to anyone in the canteen. Sunrise Bakery, Campus Canteen, Green Market Stall and Lotus Cafe could receive spam or unwanted calls, and some may stop donating.
3. **Urgent food is buried.** Alphabetical order puts the rejected bananas at the top and F004 (vegetable curry, 4 hours left, still waiting for pickup) at the bottom. A volunteer who reads from the top may miss it, and the Old Market Elders' Group may not get its 12 portions before the food spoils.
4. **Rejected food is shown as if it were available,** and there is no status column. People may ask for bananas that are not coming.

**How a test catches it:** check that no line of the public board contains `"@example.org"` (the Week 9 menu check does this); check the order of the IDs (F002 first); check the number against the records (`portions_by_status` gives delivered 18).

Version B shows one honest measure with its unit, no private fields, no rejected offers, and the fewest hours left first. **Is it perfect?** No: it still shows raw codes such as `picked_up`, and it decides alone that the public sees only delivered portions. AI dashboards often reach for one big, impressive number. Ask what it measures, and whether a person could check it.

</details>

## Session 3 · Build: `report.py` and your own public page (90 min)

*If you opened this notebook in a new session, run the setup cell at the top first so `bridge` and your `my_foodbridge/` folder are ready.*

### 📁 Your folder now

In [ ]:
bridge.show()

### Write your file

The cell below **is** your file. Its first line, `%%writefile my_foodbridge/report.py`, saves the rest of the cell as that file. Run it again after every change.

You have practised every piece: `public_row` is your `seat_row`; `most_urgent_first` is `sorted` with a key function, like `cheapest_names`; `portions_by_status` is your `total_by` with the field `"status"` and the amount `"portions"`.

In [ ]:
%%writefile my_foodbridge/report.py
# report.py · Week 8: the right information for the right person
PUBLIC_FIELDS = ["id", "food", "category", "portions", "status"]


def public_row(record):
    """Return a new dictionary with only the fields the public may see."""
    row = {}
    # TODO: for each field in PUBLIC_FIELDS, copy record.get(field, "") into row
    return row


def hours_left_of(record):
    # TODO: return the value to sort by: the record's hours left
    return 0


def most_urgent_first(records):
    """Return a new list, fewest hours left first."""
    # TODO: return sorted(records, key=...) with your key function (no brackets after its name)
    return []


def portions_by_status(records):
    """Return {status: total portions}."""
    totals = {}
    # TODO: for each record, add its portions to totals[its status]   (totals.get(status, 0) + ...)
    return totals

### Check each function

In [ ]:
bridge.check("public_row", folder="my_foodbridge")
bridge.check("most_urgent_first", folder="my_foodbridge")
bridge.check("portions_by_status", folder="my_foodbridge")

### 👀 See it

`bridge.show("board")` runs the sample day and draws both screens from **your** `report.py`: the coordinator view (every field) and the public board (only your `PUBLIC_FIELDS`), most urgent first, with a bar for each status. An unfinished part is replaced by the model version (🔧).

In [ ]:
bridge.show("board")

🤔 Read the picture as a stranger in the canteen. What can you learn from the public board? What can you **not** learn? Which offer would a volunteer see first?

📝 **My answer:** …

### 🖥️ Your own screen: the public board as a web page (⭐ everyone, 30 min)

`bridge.show` draws with the kit's code. Now **you** write a screen: about 15 lines of plain Python that turn your public rows into a web page, `my_foodbridge/public_board.html`. It is just a file (no server, no internet): open it in any browser, or send it by Telegram.

A web page is text with **tags**: `<table>` a table, `<tr>` a row, `<th>` a column heading, `<td>` a cell. A browser reads `<` as the start of a tag, so a food name with tags in it would change your page. `html.escape` turns `<`, `>` and `&` into safe codes that show as plain text.

🤔 **Predict** what is printed.

In [ ]:
import html

food = "Rice <b>FREE</b> & soup"
print(html.escape(food))

**Prediction:** …  **Result:** …

**Step 1 · Read your file.** This cell runs your `report.py` again (with `runpy`, as the kit does) and takes out three names. If the file is missing or not finished, it uses the model version and says 🔧. You do not need to change it.

In [ ]:
import runpy

model = runpy.run_path("foodbridge_kit/foodbridge/model_files/report.py")     # the model version
PUBLIC_FIELDS = model["PUBLIC_FIELDS"]
try:
    mine = runpy.run_path("my_foodbridge/report.py")     # runs your file: its names come back in a dictionary
    PUBLIC_FIELDS = mine["PUBLIC_FIELDS"]
    public_row = mine["public_row"]
    most_urgent_first = mine["most_urgent_first"]
    if public_row(bridge.sample_records()[0]) == {} or most_urgent_first(bridge.sample_records()) == []:
        raise ValueError("a function is not finished yet")
    print("📄 Your report.py makes the rows. PUBLIC_FIELDS =", PUBLIC_FIELDS)
except Exception as problem:                             # no file, a mistake, or not finished
    print("🔧 The model functions make the rows (" + type(problem).__name__ + ": " + str(problem) + ").")
    public_row = model["public_row"]
    most_urgent_first = model["most_urgent_first"]

**Step 2 · Write the page.** Fill in the three `TODO`s and run the cell. It saves the page in your folder and shows it. To keep a copy: **Files** panel → `my_foodbridge` → right-click `public_board.html` → **Download**.

In [ ]:
header = ""
for field in PUBLIC_FIELDS:
    header = header + "<th>" + html.escape(field) + "</th>"

rows = ""
for record in most_urgent_first(bridge.sample_records()):
    if record["status"] not in []:             # TODO 1: which statuses are not coming? Put them in the list
        row = public_row(record)
        cells = ""
        for field in PUBLIC_FIELDS:
            value = str(row.get(field, ""))    # TODO 2: make it safe for a web page: html.escape(...)
            cells = cells + "<td>" + value + "</td>"
        rows = rows + ""                       # TODO 3: add one table row: "<tr>" + cells + "</tr>\n"

page = ("<meta charset='utf-8'>\n<h3>Campus FoodBridge · Public board</h3>\n"
        "<table border='1'>\n<tr>" + header + "</tr>\n" + rows + "</table>\n")
with open("my_foodbridge/public_board.html", "w", encoding="utf-8") as file:
    file.write(page)
print("Saved my_foodbridge/public_board.html with", rows.count("<tr>"), "offers")
if "@" in page:
    print("🔓 A contact is on your public page. Check PUBLIC_FIELDS.")
try:
    from IPython.display import HTML, display
    display(HTML(page))                    # 👀 Colab and Jupyter draw your page here
except ImportError:
    print("Open the file in a browser to see it.")

🤔 **Change one field, watch your page.** In your file cell, remove `"category"` from `PUBLIC_FIELDS`, or add `"donor"`. **Predict** which column changes, then run the file cell and Steps 1–2 again. The page code does not change: it reads your list. (The Final project's Level 4 screen works the same way.)

**Prediction:** …  **Result:** …

⭐⭐ **Make one of your own:** a volunteer view in your file (`VOLUNTEER_FIELDS`, `volunteer_row`: what does a volunteer need that the public must not see?), or urgent rows in red on your page (`<tr style='color:red'>`).

### 🧭 Your decision · what may the public see?

Anyone can read the public board. Each extra field helps someone and may hurt someone else. Start from the Public board column of your 🔐 matrix.

| Field | Who it helps | Who it may hurt |
|---|---|---|
| `"donor"` | donors who want public thanks ("Thank you, Sunrise Bakery!") | donors who never agreed to be named |
| `"hours_left"` | partners who plan when to come | nobody much, but the board gets wider |
| `"partner"` | sponsors who want to see where food goes | groups who may not want others to know they receive food |
| `"contact"` | nobody on a public board | every donor: spam and unwanted calls. Never. |

1. Change `PUBLIC_FIELDS` at the top of your file. You may keep the default if you can explain why.
2. Run the file cell, Steps 1–2, and the cell below. The check still passes (it reads **your** list), and both boards change. A 🔓 warning appears if a private field reaches the public.

In [ ]:
bridge.check("public_row", folder="my_foodbridge")
bridge.show("board")

📝 **Decision log** — *My choice:* `PUBLIC_FIELDS = …` · *An alternative:* … · *Why I chose it:* … · *Evidence (data / week):* your 🔐 matrix, or data (every fruit offer in `semester.csv` is from the Green Market Stall, so `category` can name a donor): … · *Who is affected / the tradeoff:* who is protected, and who sees less: …

### 🌐 Optional · your board in the web app

The kit's web app reads your folder too: click the link, then the **Public board** tab. If it does not open, your page and `bridge.show("board")` show the same. After a change, run your file cell and this cell again.

In [ ]:
bridge.start_web(folder="my_foodbridge")
# Done? Run bridge.stop_web() to stop it.

### 📁 Your folder now

In [ ]:
bridge.show()

## Finish

- [ ] I shared my Research 6 check in the warm-up.
- [ ] I can explain count vs sum, why offered, picked up and delivered are never added, and what makes a chart honest.
- [ ] I filled in the 🔐 matrix and found at least three problems in the AI dashboard, with who is harmed.
- [ ] My `report.py` passes all three checks, and my own `public_board.html` shows no contact.
- [ ] My decision log explains `PUBLIC_FIELDS`, with evidence, who is protected and who sees less.

There is **no new research task**: Research 6 was the last one. Keep your decision logs from every week: the final project needs them.

**Next week:** your files meet in `my_foodbridge/app.py`, your own menu; you open your whole system in a browser, and use rates to re-decide a setting. The final project starts.

⬅ **Previous:** [Week 7 · Saving and history](Week_07.ipynb) · **Next:** [Week 9 · Assemble and analyse](Week_09.ipynb) ➡